In [19]:
import pandas as pd
from pathlib import Path

df = pd.read_parquet(Path("../data/processed/player_weeks_clean.parquet"))
df = df.sort_values(["player_id", "season", "week"]).reset_index(drop=True)
df.shape

(37313, 161)

In [20]:
cols = ["fantasy_points_ppr", "targets", "receptions", "carries", "offense_pct",
        "attempts", "passing_yards", "passing_tds", "passing_interceptions",
        "passing_epa", "rushing_yards"]

g = df.groupby(["player_id", "season"])

for c in cols:
    df[f"{c}_last3"] = g[c].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())

df[[c + "_last3" for c in cols]].describe().round(2)

,fantasy_points_ppr_last3,targets_last3,receptions_last3,carries_last3,offense_pct_last3,attempts_last3,passing_yards_last3,passing_tds_last3,passing_interceptions_last3,passing_epa_last3,rushing_yards_last3
count,33172.00,33172.00,33172.00,33172.00,33143.00,33172.00,33172.00,33172.00,33172.00,4043.00,33172.00
mean,8.00,3.00,2.03,2.50,0.53,3.11,22.14,0.14,0.07,1.04,10.87
std,6.75,2.85,1.98,4.68,0.30,9.65,69.50,0.49,0.26,6.55,21.80
min,-2.86,0.00,0.00,0.00,0.00,0.00,-3.00,0.00,0.00,-32.77,-14.00
25%,2.40,0.67,0.33,0.00,0.27,0.00,0.00,0.00,0.00,-2.68,0.00
50%,6.43,2.33,1.67,0.00,0.54,0.00,0.00,0.00,0.00,0.52,0.00
75%,12.20,4.67,3.00,2.67,0.78,0.00,0.00,0.00,0.00,4.90,10.33
max,44.50,21.00,14.00,32.00,1.00,59.00,474.67,5.00,4.00,33.19,173.00


In [21]:
show = ["player_display_name", "season", "week", "fantasy_points_ppr", "fantasy_points_ppr_last3"]
df[df["player_display_name"] == "Justin Jefferson"][show].head(6)

,player_display_name,season,week,fantasy_points_ppr,fantasy_points_ppr_last3
21667,Justin Jefferson,2020,1,4.6,NaN
21668,Justin Jefferson,2020,2,7.4,4.600000
21669,Justin Jefferson,2020,3,30.5,6.000000
21670,Justin Jefferson,2020,4,14.3,14.166667
21671,Justin Jefferson,2020,5,5.3,17.400000
21672,Justin Jefferson,2020,6,39.6,16.700000


In [22]:
# carry share: player's carries / team's carries that week
df["team_carries"] = df.groupby(["team", "season", "week"])["carries"].transform("sum")
df["carry_share"] = df["carries"] / df["team_carries"]

g = df.groupby(["player_id", "season"])      # rebuild the grouping so it sees the new column

share_cols = ["target_share", "air_yards_share", "wopr", "carry_share"]
for c in share_cols:
    df[f"{c}_last3"] = g[c].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())

df[[c + "_last3" for c in share_cols]].describe().round(2)

,target_share_last3,air_yards_share_last3,wopr_last3,carry_share_last3
count,33172.00,33172.00,33172.00,33172.00
mean,0.09,0.09,0.20,0.09
std,0.09,0.13,0.21,0.17
min,0.00,-0.39,-0.03,0.00
25%,0.02,0.00,0.04,0.00
50%,0.07,0.03,0.14,0.00
75%,0.15,0.15,0.32,0.10
max,0.59,0.97,1.52,0.98


In [23]:
# 1. total PPR points each defense allowed to each position, each week
allowed = (df.groupby(["opponent_team", "season", "week", "position"])["fantasy_points_ppr"]
             .sum().reset_index(name="ppr_allowed"))

# 2. that defense's average over all EARLIER weeks this season
allowed = allowed.sort_values(["opponent_team", "position", "season", "week"])
allowed["opp_ppr_allowed_avg"] = (allowed.groupby(["opponent_team", "position", "season"])["ppr_allowed"]
                                  .transform(lambda s: s.shift(1).expanding().mean()))

# 3. attach it to each player's row
df = df.merge(allowed[["opponent_team", "season", "week", "position", "opp_ppr_allowed_avg"]],
              on=["opponent_team", "season", "week", "position"], how="left", validate="many_to_one")

df.groupby("position")["opp_ppr_allowed_avg"].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
position,,,,,,,,
QB,3883.0,16.9,3.9,1.0,14.6,16.8,19.1,38.8
RB,9127.0,22.3,5.1,4.3,19.1,22.1,25.0,52.8
TE,7297.0,12.2,4.1,0.0,9.5,12.1,14.6,35.2
WR,14583.0,34.1,6.7,6.1,29.9,33.9,37.8,79.4


In [24]:
id_cols = ["player_id", "player_display_name", "position", "team", "opponent_team", "season", "week"]

feature_cols = [
    # player history and usage (previous 3 games)
    "fantasy_points_ppr_last3", "targets_last3", "receptions_last3",
    "carries_last3", "offense_pct_last3",
    # role (previous 3 games)
    "target_share_last3", "air_yards_share_last3", "wopr_last3", "carry_share_last3",
    # opponent (earlier weeks this season)
    "opp_ppr_allowed_avg",
    # game context (known before kickoff)
    "implied_total", "is_home", "rest_days",
    "attempts_last3", "passing_yards_last3", "passing_tds_last3",
    "passing_interceptions_last3", "passing_epa_last3", "rushing_yards_last3",
]

target = "fantasy_points_ppr"

features = df[id_cols + feature_cols + [target]]
features.to_parquet("../data/processed/features.parquet", index=False)

print(features.shape)
features[feature_cols].isna().mean().round(3)     # share of blanks in each feature

(37313, 27)


fantasy_points_ppr_last3       0.111
targets_last3                  0.111
receptions_last3               0.111
carries_last3                  0.111
offense_pct_last3              0.112
target_share_last3             0.111
air_yards_share_last3          0.111
wopr_last3                     0.111
carry_share_last3              0.111
opp_ppr_allowed_avg            0.065
implied_total                  0.000
is_home                        0.000
rest_days                      0.000
attempts_last3                 0.111
passing_yards_last3            0.111
passing_tds_last3              0.111
passing_interceptions_last3    0.111
passing_epa_last3              0.892
rushing_yards_last3            0.111
dtype: float64